# Projekt: Vorhersage der Kundenreaktion im Bankmarketing

In diesem Notebook bauen wir ein Klassifikationsmodell mit Logistic Regression, um vorherzusagen, ob ein Kunde auf eine Marketingkampagne positiv reagiert (`y = yes`) oder nicht.

## Workflow
1. Umgebung vorbereiten
2. Daten laden
3. Grundlegende Bereinigung und Kodierung
4. Features (`X`) und Zielvariable (`y`) definieren
5. Trainings- und Testdaten erstellen
6. Modell trainieren
7. Modell evaluieren
8. Wichtigste Einflussfaktoren interpretieren

Dieses Dokument ist so aufgebaut, dass sowohl der Code als auch die fachliche Begründung klar nachvollziehbar sind.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Setup abgeschlossen - Modellumgebung ist bereit")

## 1) Umgebung und Bibliotheken

Hier werden alle benötigten Bibliotheken importiert:
- `pandas`, `numpy` für Datenverarbeitung
- `train_test_split` für die Aufteilung in Train/Test
- `LogisticRegression` als Modell
- `accuracy_score`, `precision_score`, `recall_score`, `roc_auc_score` für die Bewertung
- `seaborn`, `matplotlib` für Visualisierungen

Zusätzlich werden Anzeigeoptionen für `pandas` gesetzt, damit Tabellen in der Ausgabe besser lesbar sind.

In [ ]:
# --- Bank-Marketing-Datensatz laden (Additional Full) ---

file_path = r"C:\Users\shiva\OneDrive\Desktop\weiter bildung\projekt\ki-marketing-personalisierung-main\data\raw\bank_marketing\bank-additional-full.csv"

df = pd.read_csv(file_path, sep=';')

print("Datensatz wurde erfolgreich geladen")
df.head()

## 2) Datensatz laden

Der Datensatz `bank-additional-full.csv` wird mit `sep=';'` geladen, da die Datei semikolon-separiert ist.

`df.head()` zeigt die ersten Zeilen zur schnellen Plausibilitätsprüfung.

Empfehlung für eine vollständige Datenprüfung:
- `df.shape` für Dimensionen
- `df.info()` für Datentypen
- `df.isna().sum()` für fehlende Werte

In [ ]:
# --- Grundlegende Bereinigung und kategoriale Kodierung ---

df_clean = df.copy()

for col in df_clean.select_dtypes(include=['object']).columns:
    df_clean[col] = df_clean[col].astype('category').cat.codes

print("Bereinigung und Kodierung abgeschlossen")
df_clean.head()

## 3) Datenbereinigung und Kodierung kategorialer Merkmale

Da Logistic Regression numerische Eingaben benötigt, werden alle Spalten vom Typ `object` in numerische Codes umgewandelt.

Ablauf:
- Mit `df.copy()` wird eine Arbeitskopie erstellt (`df_clean`), damit Rohdaten unverändert bleiben.
- Für jede kategoriale Spalte wird `astype('category').cat.codes` verwendet.

Hinweis: Diese Kodierung ist schnell, aber nicht immer ideal für Interpretation. Für viele Anwendungsfälle ist One-Hot-Encoding (`pd.get_dummies`) die robustere Alternative.

In [ ]:
# --- Schritt 2: Merkmale und Zielvariable definieren ---

X = df_clean.drop("y", axis=1)
y = df_clean["y"]

print("Schritt 2 abgeschlossen - X und y sind vorbereitet")
X.head()
y.head()

## 4) Features und Zielvariable definieren

In diesem Schritt werden Eingabemerkmale und Ziel getrennt:
- `X`: alle Prädiktoren (alle Spalten außer `y`)
- `y`: Zielvariable (Kundenreaktion)

Diese Trennung ist die Standardbasis für überwachtes Lernen.

In [ ]:
# --- Schritt 3: Aufteilung in Training und Test ---

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Schritt 3 abgeschlossen - Daten in Trainings- und Testmenge aufgeteilt")

## 5) Aufteilung in Trainings- und Testdaten

Mit `train_test_split` wird der Datensatz in zwei Teile getrennt:
- Training: zum Lernen der Modellparameter
- Test: zur objektiven Bewertung auf ungesehenen Daten

Wichtige Parameter:
- `test_size=0.25`: 25 % Testanteil
- `random_state=42`: reproduzierbare Ergebnisse
- `stratify=y`: Klassenverhältnis bleibt in beiden Teilmengen erhalten

In [ ]:
# --- Schritt 4: Logistic-Regression-Modell trainieren ---

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

print("Schritt 4 abgeschlossen - Pipeline mit Skalierung und Logistic Regression wurde trainiert")

## 6) Logistic-Regression-Modell trainieren

Das Modell wird als Pipeline trainiert:
- `StandardScaler()` zur Skalierung numerischer Merkmale
- `LogisticRegression(max_iter=1000, random_state=42)` für die Klassifikation

Warum diese Anpassung?
- Skalierung verbessert die numerische Stabilität des Optimierers.
- Höhere Iterationszahl reduziert das Risiko von Konvergenz-Warnungen.
- Das Modell bleibt gut interpretierbar über die Koeffizienten.

Nach `fit(...)` hat die Pipeline die Beziehung zwischen Merkmalen und Ziel gelernt.

In [ ]:
# --- Schritt 5: Bewertungsmetriken berechnen ---

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Genauigkeit:", accuracy_score(y_test, y_pred))
print("Praezision:", precision_score(y_test, y_pred))
print("Trefferquote (Recall):", recall_score(y_test, y_pred))
print("Flaeche unter der ROC-Kurve (AUC):", roc_auc_score(y_test, y_prob))

print("Schritt 5 abgeschlossen - Evaluationsmetriken wurden berechnet")

## 7) Modellbewertung

Es werden zwei zentrale Ausgaben erzeugt:
- `y_pred`: vorhergesagte Klassen
- `y_prob`: Wahrscheinlichkeit für die positive Klasse

Berechnete Metriken:
- **Accuracy**: Anteil korrekter Vorhersagen
- **Precision**: Anteil echter Positiver unter allen positiv vorhergesagten Fällen
- **Recall**: Anteil erkannter Positiver unter allen tatsächlich positiven Fällen
- **AUC**: Trennfähigkeit des Modells über alle Schwellenwerte hinweg

Interpretation im Geschäftskontext:
- Fokus auf Precision, wenn Fehlkontakte teuer sind.
- Fokus auf Recall, wenn verpasste Chancen kritisch sind.

In [ ]:
# --- Schritt 6: Einfluss der Merkmale bestimmen ---

logreg = model.named_steps["logisticregression"]

importance = pd.DataFrame({
    "feature": X.columns,
    "coef": logreg.coef_[0]
}).sort_values(by="coef", ascending=False)

importance.head(15)

## 8) Einfluss der Merkmale (Feature Importance)

Bei Logistic Regression geben die Koeffizienten Richtung und Stärke des Einflusses auf die positive Klasse an:
- positiver Koeffizient: erhöht die Wahrscheinlichkeit einer positiven Reaktion
- negativer Koeffizient: senkt die Wahrscheinlichkeit

Mit `importance.head(15)` werden die wichtigsten Merkmale angezeigt.

Hinweis: Koeffizienten sind besser vergleichbar, wenn numerische Features vorher skaliert wurden (z. B. Standardisierung).

## 9) Fazit und naechste Schritte

### Fazit
Das Modell liefert eine nachvollziehbare erste Grundlage zur Vorhersage von Kundenreaktionen in Marketingkampagnen.
Die ausgegebenen Kennzahlen helfen dabei, die Modellqualitaet aus fachlicher Sicht zu bewerten.

### Naechste Schritte
1. Konfusionsmatrix und ROC-Kurve visualisieren, um die Modellleistung detaillierter zu analysieren.
2. Hyperparameter-Optimierung (z. B. `C`, `solver`) mit Cross-Validation durchfuehren.
3. One-Hot-Encoding und Feature-Skalierung testen, um Robustheit und Interpretierbarkeit weiter zu verbessern.
4. Das trainierte Modell speichern (z. B. mit `joblib`), um es spaeter in einer Anwendung wiederzuverwenden.